In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
import pandas as pd
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import matplotlib.pyplot as plt

In [41]:
mh_data = pd.read_csv('mental_health.csv', sep=";")
mh_data.head()

,Entity,Code,Year,schizophrenia,depressive_disorder,anxiety_disorders,bipolar_disorders,eating_disorders,unemployment_rate,co2_emissions,gdp
0,Afghanistan,AFG,1990,0.223206,4.996118,4.713314,0.703023,0.127700,NaN,2.8965,NaN
1,Afghanistan,AFG,1991,0.222454,4.989290,4.702100,0.702069,0.123256,7.946,2.7663,NaN
2,Afghanistan,AFG,1992,0.221751,4.981346,4.683743,0.700792,0.118844,7.940,1.6826,NaN
3,Afghanistan,AFG,1993,0.220987,4.976958,4.673549,0.700087,0.115089,7.961,1.6083,NaN
4,Afghanistan,AFG,1994,0.220183,4.977782,4.670810,0.699898,0.111815,7.980,1.5358,NaN


In [42]:
disorders = ["schizophrenia", "depressive_disorder", "anxiety_disorders", "bipolar_disorders", "eating_disorders", "co2_emissions", "gdp"]

In [43]:
def classify_disorders(df, disorders):
    """
    Classify the values of multiple disorders into categories 'A', 'B', or 'C' based on their percentage.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    disorders (list): A list of disorder column names to process.

    Returns:
    None: The function directly modifies the given DataFrame.
    """
    # Create a copy of the DataFrame to avoid SettingWithCopyWarning
    df_copy = df.copy()

    for disorder in disorders:
        classification_column = f"{disorder}_classification"
        max_value = df_copy[disorder].max()

        # Use .loc to safely assign values to a new column
        df.loc[:, classification_column] = df_copy[disorder].apply(
            lambda x: classify_percentage((x / max_value) * 100 if pd.notna(x) else None)
        )

def classify_percentage(percentage):
    """
    Classify a given percentage into one of three categories: 'A', 'B', or 'C'.

    Parameters:
    percentage (float): The input percentage to be classified.

    Returns:
    str: A single character representing the classification ('A', 'B', or 'C').
    """
    if percentage is None:  # Handle NaN values
        return None
    elif percentage < 100 / 3:
        return 'A'
    elif percentage < 100 * 2 / 3:
        return 'B'
    else:
        return 'C'

# Explore data

### Define lists of blended colors 

To get bivariate maps working, we need a set of nine colors that are the result of blending two main colors. Here are some examples, but others can easily be added:

1) "pink-blue" by [Joshua Stevens](http://www.joshuastevens.net/cartography/make-a-bivariate-choropleth-map/)
![Three examples of color sets](https://raw.githubusercontent.com/yotkadata/plotly-bivariate-choropleth/main/img/colors.png)

In [44]:
# Define sets of 9 colors to be used
# Order: bottom-left, bottom-center, bottom-right, center-left, center-center, center-right, top-left, top-center, top-right
color_sets = {
    'pink-blue':   ['#e8e8e8', '#ace4e4', '#5ac8c8', '#dfb0d6', '#a5add3', '#5698b9', '#be64ac', '#8c62aa', '#3b4994'],
    'teal-red':    ['#e8e8e8', '#e4acac', '#c85a5a', '#b0d5df', '#ad9ea5', '#985356', '#64acbe', '#627f8c', '#574249'],
    'blue-organe': ['#fef1e4', '#fab186', '#f3742d',  '#97d0e7', '#b0988c', '#ab5f37', '#18aee5', '#407b8f', '#5c473d']
}

In [45]:
import plotly.graph_objects as go

def add_bivariate_legend(fig, x_legend, y_legend, colors, conf=None):
    """
    Add a bivariate choropleth coddlor legend to a Plotly figure.

    Parameters:
    - fig (plotly.graph_objects.Figure): The Plotly figure to which the legend will be added.
    - colors (list): A list of 9 colors representing the bivariate legend. The colors should be ordered from low-low to high-high.
    - conf (dict, optional): Configuration dictionary for legend customization. Defaults are used if not provided.

    Returns:
    - fig (plotly.graph_objects.Figure): The updated Plotly figure with the legend added.
    """
    # Use default configuration if none is provided
    if conf is None:
        conf = {
            'top': 0.3,  # Vertical position of the top right corner (0: bottom, 1: top)
            'right': 0.2,  # Horizontal position of the top right corner (0: left, 1: right)
            'box_w': 0.04,  # Width of each rectangle
            'box_h': 0.08,  # Height of each rectangle
            'line_color': 'rgba(0,0,0,0)',  # Transparent borders
            'line_width': 0,  # Width of the rectangle borders
            'legend_x_label': f'{x_legend}'+  '→',  # Label for the x-axis
            'legend_y_label': f'{y_legend}' '→',  # Label for the y-axis
            'legend_font_size': 14,  # Font size for the legend text
            'legend_font_color': '#000',  # Font color for the legend text
        }

    # Reverse the order of colors for correct display
    legend_colors = colors[:]
    legend_colors.reverse()

    # Calculate coordinates for all nine rectangles
    coord = []
    width = conf['box_w']
    height = conf['box_h']

    for row in range(1, 4):  # 3 rows
        for col in range(1, 4):  # 3 columns
            coord.append({
                'x0': round(conf['right'] - (col - 1) * width, 4),
                'y0': round(conf['top'] - (row - 1) * height, 4),
                'x1': round(conf['right'] - col * width, 4),
                'y1': round(conf['top'] - row * height, 4)
            })

    # Create rectangles and add to the figure
    for i, value in enumerate(coord):
        fig.add_shape(
            type='rect',
            x0=value['x0'], y0=value['y0'], x1=value['x1'], y1=value['y1'],
            xref='paper', yref='paper',
            fillcolor=legend_colors[i],
            line=dict(
                color=conf['line_color'],
                width=conf['line_width']
            )
        )

    # Add x-axis legend label
    fig.add_annotation(
        x=coord[8]['x1'], y=coord[8]['y1'],  #position
        xref='paper', yref='paper',
        showarrow=False,
        text=f"{conf['legend_x_label']} 🠒",
        font=dict(
            size=conf['legend_font_size'],
            color=conf['legend_font_color']
        ),
        xanchor='left', yanchor='top',
        borderpad=0
    )

    # Add y-axis legend label
    fig.add_annotation(
        x=coord[8]['x1'], y=coord[8]['y1'],  #position
        xref='paper', yref='paper',
        showarrow=False,
        text=f"{conf['legend_y_label']}",
        font=dict(
            size=conf['legend_font_size'],
            color=conf['legend_font_color']
        ),
        textangle=270,
        xanchor='right', yanchor='bottom',
        borderpad=0
    )

    return fig

In [46]:
# Mapping classifications to colors based on bivariate legend
def create_bivariate_color_mapping(colors):
    return {
        ('A', 'A'): colors[0],  # Bottom-left
        ('B', 'A'): colors[1],  # Middle-left
        ('C', 'A'): colors[2],  # Top-left
        ('A', 'B'): colors[3],  # Bottom-center
        ('B', 'B'): colors[4],  # Center
        ('C', 'B'): colors[5],  # Top-center
        ('A', 'C'): colors[6],  # Bottom-right
        ('B', 'C'): colors[7],  # Middle-right
        ('C', 'C'): colors[8],  # Top-right
    }

# Assign colors to countries based on their classifications
def assign_bivariate_colors(df, disorder, factor, color_mapping):
    df['color'] = df.apply(
        lambda row: color_mapping.get(
            (row[f"{disorder}_classification"], row[f"{factor}_classification"]),
            'gray'  # Default color if classification is missing
        ),
        axis=1
    )
    return df

import plotly.express as px

def plot_bivariate_map(df, disorder, factor, color_set_name, color_sets):
    """
    Plot a bivariate choropleth map based on classifications.

    Parameters:
    - df (pd.DataFrame): The DataFrame with classification and color data.
    - disorder (str): The disorder to classify.
    - factor (str): The factor to classify.
    - color_set_name (str): The name of the color set to use (e.g., 'pink-blue').
    - color_sets (dict): A dictionary of color sets.

    Returns:
    - go.Figure: The Plotly figure.
    """
    # Select the color set
    colors = color_sets[color_set_name]

    # Create a color mapping
    color_mapping = create_bivariate_color_mapping(colors)

    # Assign colors to countries
    df = assign_bivariate_colors(df, disorder, factor, color_mapping)

    # Create the choropleth map
    fig = px.choropleth(
        df,
        locations="Code",  # ISO-3 country codes
        color="color",  # Column with the assigned bivariate colors
        hover_data={
        disorder: ':.2f',  # Show disorder value with 2 decimal places
        factor: ':.2f',    # Show factor value with 2 decimal places
        "Code": True},   # Optionally hide the country code,  # Country names for hover info
        title=f"Bivariate Map of {disorder.replace('_', ' ').title()} and {factor.replace('_', ' ').title()}",
        color_discrete_map="identity"  # Use exact color mapping from the DataFrame
    )

    # Update the map layout
    fig.update_geos(
        showcoastlines=True,
        coastlinecolor="Black",
        showland=True,
        landcolor="lightgray",
    )
    fig.update_layout(margin={"r": 0, "t": 30, "l": 0, "b": 0})

    # Add the bivariate legend
    add_bivariate_legend(fig, factor.replace('_', ' ').title() ,disorder.replace('_', ' ').title(), colors)

    return fig

In [47]:
df = mh_data
disorders_and_factors = ["schizophrenia", "depressive_disorder", "anxiety_disorders", "bipolar_disorders", "eating_disorders","unemployment_rate" ,"co2_emissions", "gdp"]
classify_disorders(df, disorders=disorders_and_factors)

df.head()


,Entity,Code,Year,schizophrenia,depressive_disorder,anxiety_disorders,bipolar_disorders,eating_disorders,unemployment_rate,co2_emissions,gdp,schizophrenia_classification,depressive_disorder_classification,anxiety_disorders_classification,bipolar_disorders_classification,eating_disorders_classification,unemployment_rate_classification,co2_emissions_classification,gdp_classification
0,Afghanistan,AFG,1990,0.223206,4.996118,4.713314,0.703023,0.127700,NaN,2.8965,NaN,B,B,B,B,A,None,A,None
1,Afghanistan,AFG,1991,0.222454,4.989290,4.702100,0.702069,0.123256,7.946,2.7663,NaN,B,B,B,B,A,A,A,None
2,Afghanistan,AFG,1992,0.221751,4.981346,4.683743,0.700792,0.118844,7.940,1.6826,NaN,B,B,B,B,A,A,A,None
3,Afghanistan,AFG,1993,0.220987,4.976958,4.673549,0.700087,0.115089,7.961,1.6083,NaN,B,B,B,B,A,A,A,None
4,Afghanistan,AFG,1994,0.220183,4.977782,4.670810,0.699898,0.111815,7.980,1.5358,NaN,B,B,B,B,A,A,A,None


In [48]:
from dash import Dash, dcc, html, Input, Output, callback

disorders_and_factors = ["schizophrenia", "depressive_disorder", "anxiety_disorders", "bipolar_disorders","unemployment_rate", "eating_disorders", "co2_emissions", "gdp"]

# Select specific columns for the dropdown-y
disorders_to_include = [
    'schizophrenia',
    'depressive_disorder',
    'anxiety_disorders',
    'bipolar_disorders',
    'eating_disorders',]
dropdown_options_disorder = [{'label': disorder, 'value': disorder} for disorder in disorders_to_include]

factors_to_include = [
    'co2_emissions',
    'gdp',
    'unemployment_rate',
]

dropdown_options_factor = [{'label': factor, 'value': factor} for factor in factors_to_include]


app = Dash()

app.layout = html.Div([


    html.H3("Change the value in the text box to see callbacks in action!"),

    dcc.Slider(
        df['Year'].min(),
        df['Year'].max(),
        step=None,
        id='year--slider',
        value=df['Year'].max(),
        marks={str(year): str(year) for year in df['Year'].unique()},
    ),

    dcc.Dropdown(
        id='y-disorder-dropdown',
        options=dropdown_options_disorder,
        value=disorders_to_include[0],  # Default value
        clearable=False
    ),
    html.Div(id='output-div-disorder'),

    dcc.Dropdown(
        id='x-factor-dropdown',
        options=dropdown_options_factor,
        value=factors_to_include[1],  # Default value
        clearable=False
    ),
    html.Div(id='output-div-factor'),

    dcc.Graph(id='bivariate-map'),

    html.Div(id='selected-country-output', style={"marginTop": "20px"})
])


@app.callback(
    Output('bivariate-map', 'figure'),
    [Input('y-disorder-dropdown', 'value'),
     Input('x-factor-dropdown', 'value'),
     Input('year--slider', 'value')]
)
def update_output_div(selected_disorder, selected_factor, selected_year):
    dff = df.copy()
    filtered_dff = dff[dff['Year'] == selected_year].copy()

    try:
        # Use classification column names for plotting
        fig = plot_bivariate_map(
            filtered_dff,
            selected_disorder,
            selected_factor,
            "pink-blue",
            color_sets
        )
        return fig
    except Exception as e:
        print(f"Error in update_output_div: {e}")
        return {}

@app.callback(
    Output('selected-country-output', 'children'),
    [Input('bivariate-map', 'clickData')]
)
def display_selected_country(click_data):
    if click_data is None:
        return "Click on a country on the map to see its code."

    # Extract the country Code from the clickData
    country_code = click_data['points'][0]['location']  # 'location' refers to the `Code` in the scatter_geo
    print(f"Selected Country Code: {country_code}")  # In die Konsole ausgeben
    return f"Selected Country Code: {country_code}"






if __name__ == '__main__':
    app.run(debug=True)

In [49]:
from dash import Dash, dcc, html, Input, Output

disorders_and_factors = ["schizophrenia", "depressive_disorder", "anxiety_disorders", "bipolar_disorders",
                         "unemployment_rate", "eating_disorders", "co2_emissions", "gdp"]

# Select specific columns for the dropdown-y
disorders_to_include = [
    'schizophrenia',
    'depressive_disorder',
    'anxiety_disorders',
    'bipolar_disorders',
    'eating_disorders']
dropdown_options_disorder = [{'label': disorder, 'value': disorder} for disorder in disorders_to_include]

factors_to_include = [
    'co2_emissions',
    'gdp',
    'unemployment_rate',
]
dropdown_options_factor = [{'label': factor, 'value': factor} for factor in factors_to_include]

app = Dash()

app.layout = html.Div([

    html.H3("Change the value in the text box to see callbacks in action!"),

    dcc.Slider(
        df['Year'].min(),
        df['Year'].max(),
        step=None,
        id='year--slider',
        value=df['Year'].max(),
        marks={str(year): str(year) for year in df['Year'].unique()},
    ),

    dcc.Dropdown(
        id='y-disorder-dropdown',
        options=dropdown_options_disorder,
        value=disorders_to_include[0],  # Default value
        clearable=False
    ),
    html.Div(id='output-div-disorder'),

    dcc.Dropdown(
        id='x-factor-dropdown',
        options=dropdown_options_factor,
        value=factors_to_include[1],  # Default value
        clearable=False
    ),
    html.Div(id='output-div-factor'),

    dcc.Graph(id='bivariate-map'),

    html.Div(id='selected-country-output', style={"marginTop": "20px"})
])

@app.callback(
    [Output('bivariate-map', 'figure'),
     Output('selected-country-output', 'children')],
    [Input('bivariate-map', 'clickData'),
     Input('y-disorder-dropdown', 'value'),
     Input('x-factor-dropdown', 'value'),
     Input('year--slider', 'value')]
)
def update_map_with_highlight(click_data, selected_disorder, selected_factor, selected_year):
    dff = df.copy()
    filtered_dff = dff[dff['Year'] == selected_year].copy()

    # Determine the clicked country
    clicked_country_code = None
    if click_data:
        clicked_country_code = click_data['points'][0]['location']
        print(f"Clicked Country Code: {clicked_country_code}")

    try:
        # Use classification column names for plotting
        fig = plot_bivariate_map(
            filtered_dff,
            selected_disorder,
            selected_factor,
            "pink-blue",
            color_sets
        )

        # Highlight the clicked country in orange
        if clicked_country_code:
            fig.add_trace(px.choropleth(
                filtered_dff[filtered_dff['Code'] == clicked_country_code],
                locations="Code",
                locationmode="ISO-3",
                color_discrete_sequence=["orange"],  # Highlight color
                hover_name="Code"
            ).data[0])

        # Return the updated figure and selected country code
        country_output = f"Selected Country Code: {clicked_country_code}" if clicked_country_code else "No country selected"
        return fig, country_output
    except Exception as e:
        print(f"Error in update_map_with_highlight: {e}")
        return {}, "Error: Could not update the map."


if __name__ == '__main__':
    app.run(debug=True)

In [ ]:
def plot_default_map(df):
    """
    Plot a default choropleth map without any bivariate coloring.

    Parameters:
    - df (pd.DataFrame): The DataFrame with country data.

    Returns:
    - go.Figure: The Plotly figure with default map styling.
    """
    # Add a default color column to ensure all countries are the same color
    df['default_color'] = 'lightgray'  # Assign a neutral color

    fig = px.choropleth(
        df,
        locations="Code",  # ISO-3 country codes
        color="default_color",  # Use the default color
        color_discrete_map={'lightgray': 'lightgray'},  # Map lightgray as the color
        hover_data={
            "Entity": False  # Show country names on hover (if available in the DataFrame)
        },
        title="Default Map (Select a Country to Highlight)"
    )

    # Update the map layout
    fig.update_geos(
        showcoastlines=True,
        coastlinecolor="Black",
        showland=True,
        landcolor="white",  # Set the land to white for a clean background
    )
    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
    )

    return fig

In [54]:
plot_default_map(df)